# LGBM Regression Training - Spearman First

Train deployable LightGBM regression models for the long, short, and signed quality targets.
The notebook uses purge-aware walk-forward validation, tunes by Spearman, and exports a model pack
that can be consumed by the production stack.

In [1]:
import inspect
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

from Learn.features import _add_features_US500_v2, select_features_pipeline
from Learn.labels import calculate_trade_outcomes_capped, generate_tradeability_scores, resolution_rate_by_horizon
from Learn.preprocess import preprocess_ohlcv
from Learn.train_utils import (
    load_ohlcv, safe_spearman, reg_metrics,
    walk_forward_folds, describe_target_frame, tune_lgbm_by_spearman,
)

warnings.filterwarnings("ignore")

In [2]:
# ========== CORE CONFIGURATION ==========
DS_NAME = "../data/EURUSD_M1_520weeks.csv"
N_ROWS = 500_000

# --- Asset-specific settings ---
TP_MULT = 3.0
SL_MULT = 2.0
TARGET_CLIP_MAX = max(TP_MULT, SL_MULT) * 1.5
TRAIN_FRACTION = 0.90

# --- Walk-forward validation ---
MAX_TUNING_ROWS = 350_000
WFO_FOLDS = 4
WFO_MIN_TRAIN_FRAC = 0.55
WFO_TEST_FRAC = 0.10
WFO_GAP_ROWS = max(60, 30 * 2)  # At least 60 bars purge gap
EARLY_STOPPING_ROUNDS = 200
ROBUSTNESS_PENALTY = 0.25

# --- Feature engineering ---
USE_FEATURE_SELECTION = True
SELECTED_FEATURE_COUNT = 80

# --- Tradeability weighting ---
USE_TRADEABILITY_WEIGHTING = True
TRADEABILITY_N_FOLDS = 5
TRADEABILITY_MIN_TRAIN = 100_000
TRADEABILITY_TEST_SIZE = 50_000

# --- Targets ---
TARGET_COLS = ["long_quality", "short_quality", "signed_quality"]
CLASSIFICATION_TARGETS = ["buy_win", "sell_win", "signed_win"]
PRIMARY_TARGET = "signed_quality"
TIME_PENALTY_LAMBDA = 0.1  # NOTE: no longer applied to targets (see two-head label redesign); kept for backward compatibility

# --- Horizon calibration (picks outcome_params['max_horizon']) ---
MAX_HORIZON_CANDIDATES = [20, 30, 45, 60]
TARGET_RESOLUTION_RATE = 0.90

# --- Optional model enhancements ---
TRAIN_CLASSIFIER = True
ENSEMBLE_SIZE = 5
ENSEMBLE_SEEDS = [42, 123, 456, 789, 1111]

# --- Regime params ---
regime_params = {
    "ma_period": 90,
    "slope_smoothness": 50,
    "regime_min_duration": 0,
    "atr_window": 60,
    "atr_lookback": 720,
    "atr_percentile": 0.0,
    "slope_threshold": 0,
}

# --- Outcome params (must match backtest) ---
outcome_params = {
    "atr_window": 60,
    "tp_mult": TP_MULT,
    "sl_mult": SL_MULT,
    "max_horizon": 30,
}

# --- LGBM base params ---
LGBM_BASE_PARAMS = {
    "objective": "regression",
    "boosting_type": "gbdt",
    "n_estimators": 5000,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
    "max_depth": -1,
    "subsample_freq": 1,
}

LGBM_CANDIDATES = [
    dict(learning_rate=0.03, num_leaves=63, min_child_samples=60, subsample=0.80, colsample_bytree=0.80, reg_alpha=0.10, reg_lambda=0.50, min_split_gain=0.00),
    dict(learning_rate=0.02, num_leaves=127, min_child_samples=80, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.20, reg_lambda=1.00, min_split_gain=0.00),
    dict(learning_rate=0.01, num_leaves=255, min_child_samples=120, subsample=0.90, colsample_bytree=0.90, reg_alpha=0.50, reg_lambda=3.00, min_split_gain=0.00),
    dict(learning_rate=0.05, num_leaves=31, min_child_samples=100, subsample=0.75, colsample_bytree=0.75, reg_alpha=0.05, reg_lambda=5.00, min_split_gain=0.00),
    dict(learning_rate=0.025, num_leaves=63, min_child_samples=40, subsample=0.85, colsample_bytree=0.80, reg_alpha=0.10, reg_lambda=2.00, min_split_gain=0.01),
    dict(learning_rate=0.02, num_leaves=95, min_child_samples=50, subsample=0.80, colsample_bytree=0.75, reg_alpha=0.00, reg_lambda=1.00, min_split_gain=0.00),
]

# Select feature function based on whether we use automated selection
if USE_FEATURE_SELECTION:
    def FEATURES(df, include_mtf=True, regime_params=None):
        return _add_features_US500_v2(
            df, include_mtf=include_mtf, regime_params=regime_params
        )
else:
    FEATURES = _add_features_US500

print("Device: CPU (tree model)")
print("Dataset:", DS_NAME)
print("Targets:", TARGET_COLS)
print("Feature function:", getattr(FEATURES, "__name__", str(FEATURES)))
print(f"R:R = {TP_MULT/SL_MULT:.2f} (TP_MULT={TP_MULT}, SL_MULT={SL_MULT})")
print(f"Purge gap: {WFO_GAP_ROWS} bars")

Device: CPU (tree model)
Dataset: ../data/EURUSD_M1_520weeks.csv
Targets: ['long_quality', 'short_quality', 'signed_quality']
Feature function: FEATURES
R:R = 1.50 (TP_MULT=3.0, SL_MULT=2.0)
Purge gap: 60 bars


In [3]:
# Calibrate max_horizon: pick the smallest candidate that reaches TARGET_RESOLUTION_RATE
# for both buy and sell sides, so the resolved (TP/SL) fraction feeding the magnitude
# regressor is large and the classifier's "timeout" class doesn't dominate.
_df_calib = load_ohlcv(DS_NAME, n_rows=N_ROWS)
horizon_table = resolution_rate_by_horizon(
    _df_calib,
    atr_window=outcome_params["atr_window"],
    tp_mult=outcome_params["tp_mult"],
    sl_mult=outcome_params["sl_mult"],
    horizon_candidates=MAX_HORIZON_CANDIDATES,
)
display(horizon_table)

_qualifying = horizon_table[
    (horizon_table["buy_resolved_rate"] >= TARGET_RESOLUTION_RATE)
    & (horizon_table["sell_resolved_rate"] >= TARGET_RESOLUTION_RATE)
]
if len(_qualifying) > 0:
    chosen_horizon = int(_qualifying.index.min())
else:
    chosen_horizon = int(horizon_table.index.max())
    print(
        f"WARNING: no candidate horizon reached TARGET_RESOLUTION_RATE={TARGET_RESOLUTION_RATE}; "
        f"falling back to the largest candidate ({chosen_horizon})."
    )

outcome_params["max_horizon"] = chosen_horizon
print(f"Calibrated max_horizon = {chosen_horizon} bars")
del _df_calib

,buy_resolved_rate,sell_resolved_rate
horizon,,
20,0.774186,0.778082
30,0.875252,0.880508
45,0.944556,0.946908
60,0.972628,0.974000


Calibrated max_horizon = 45 bars


In [ ]:
# Load data and compute target variables
from talib import ATR

df = load_ohlcv(DS_NAME, n_rows=N_ROWS)
print(f"Loaded {len(df):,} rows")
print(df["Time"].min(), "->", df["Time"].max())

# Step 1: Generate features
df = FEATURES(df.copy(), include_mtf=True, regime_params=regime_params)
print(f'Added features: {len(df.columns)} columns')

# Step 2: Feature selection (if enabled)
if USE_FEATURE_SELECTION:
    print("Running automated feature selection...")
    # Compute targets first on the full feature set for selection
    df_for_selection = df.copy()
    df_for_selection["atr"] = ATR(df_for_selection["High"], df_for_selection["Low"], df_for_selection["Close"], timeperiod=14)
    outcomes_fs = calculate_trade_outcomes_capped(df_for_selection, **outcome_params)
    eps_fs = 1e-8
    long_q_fs = np.log1p(outcomes_fs["buy_MFE"] / (outcomes_fs["buy_MAE"] + eps_fs))
    short_q_fs = np.log1p(outcomes_fs["sell_MFE"] / (outcomes_fs["sell_MAE"] + eps_fs))
    if TARGET_CLIP_MAX is not None:
        long_q_fs = np.clip(long_q_fs, 0.0, TARGET_CLIP_MAX)
        short_q_fs = np.clip(short_q_fs, 0.0, TARGET_CLIP_MAX)
    df_for_selection["signed_quality"] = long_q_fs - short_q_fs
    
    # Use a subset for speed (last 500k rows)
    fs_subset = df_for_selection.tail(min(500_000, len(df_for_selection)))
    selected_features = select_features_pipeline(
        fs_subset,
        target_col='signed_quality',
        n_features=SELECTED_FEATURE_COUNT,
        correlation_threshold=0.95,
        include_mtf=True,
        regime_params=regime_params,
    )
    print(f"Selected {len(selected_features)} features")
    
    # Re-generate features from FRESH raw data with only selected columns.
    # IMPORTANT: Use the original raw df (before any feature engineering) to avoid
    # _x/_y suffixed duplicate columns that occur when _add_features_US500_v2 is
    # called on a dataframe that already has MTF columns.
    df_raw_fresh = load_ohlcv(DS_NAME, n_rows=N_ROWS)
    df = _add_features_US500_v2(
        df_raw_fresh, include_mtf=True, regime_params=regime_params,
        selected_features=selected_features
    )
    print(f"Feature matrix: {len(df)} rows x {len(selected_features)} features")
    
    # Update FEATURES for model pack export (single-pass, uses selected_features)
    def FEATURES(df, include_mtf=True, regime_params=None):
        return _add_features_US500_v2(
            df, include_mtf=include_mtf, regime_params=regime_params,
            selected_features=selected_features
        )
else:
    # Still need ATR for non-selection path
    df["atr"] = ATR(df["High"], df["Low"], df["Close"], timeperiod=14)
    print('Added ATR')

# Step 3: Compute ATR (needed for label computation)
df["atr"] = ATR(df["High"], df["Low"], df["Close"], timeperiod=14)

# Step 4: Compute trade outcomes using horizon-capped label function
outcomes = calculate_trade_outcomes_capped(df, **outcome_params)
print(f"Calculated trade outcomes for {len(outcomes):,} rows")

eps = 1e-8
long_q = np.log1p(outcomes["buy_MFE"] / (outcomes["buy_MAE"] + eps))
short_q = np.log1p(outcomes["sell_MFE"] / (outcomes["sell_MAE"] + eps))
print(f"Computed long and short quality metrics for {len(long_q):,} rows")

if TARGET_CLIP_MAX is not None:
    long_q = np.clip(long_q, 0.0, TARGET_CLIP_MAX)
    short_q = np.clip(short_q, 0.0, TARGET_CLIP_MAX)
print(f"Clipped long and short quality metrics to max {TARGET_CLIP_MAX}")

df["long_quality"] = long_q
df["short_quality"] = short_q
df["signed_quality"] = df["long_quality"] - df["short_quality"]

# 3-class win/loss/timeout labels (1=TP, 0=SL, 2=timeout/unresolved within max_horizon).
# The magnitude regressor (long_quality/short_quality) is trained only on resolved rows
# (buy_class != 2 / sell_class != 2) later in the preprocessing step; the classifier is
# trained on all rows including timeouts as the third class.
df["buy_class"] = outcomes["buy_class"].values
df["sell_class"] = outcomes["sell_class"].values

# Binary classification targets
df['buy_win'] = (outcomes['buy_outcome'] == 1.0).astype(float)
df['sell_win'] = (outcomes['sell_outcome'] == 1.0).astype(float)
df['signed_win'] = np.where(
    df['signed_quality'] > 0, df['buy_win'],
    np.where(df['signed_quality'] < 0, df['sell_win'], 0.0)
)

print("Target summary:")
display(describe_target_frame(df, TARGET_COLS))

print(
    f"Zero share | long={(df['long_quality'] <= 1e-12).mean():.3f} "
    f"short={(df['short_quality'] <= 1e-12).mean():.3f} "
    f"signed={(df['signed_quality'] == 0).mean():.3f}"
)
print(
    f"Class balance | buy TP/SL/timeout={df['buy_class'].value_counts(normalize=True).sort_index().round(3).to_dict()} "
    f"sell TP/SL/timeout={df['sell_class'].value_counts(normalize=True).sort_index().round(3).to_dict()}"
)

Loaded 500,000 rows
2025-02-14 00:20:00+00:00 -> 2026-06-19 20:54:00+00:00
Added features: 203 columns
Running automated feature selection...


In [ ]:
# Generate Tradeability Scores
if USE_TRADEABILITY_WEIGHTING:
    print("Generating tradeability scores...")
    tradeability_y = pd.Series(
        (~outcomes['buy_outcome'].isna() | ~outcomes['sell_outcome'].isna()).astype(int),
        name='tradeable'
    )
    
    # Use top features by variance for the tradeability classifier
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    exclude_cols = set(TARGET_COLS + CLASSIFICATION_TARGETS + ['tradeability_score', 'atr'])
    feature_candidates = [c for c in numeric_cols if c not in exclude_cols]
    
    if len(feature_candidates) > 50:
        variances = df[feature_candidates].var().sort_values(ascending=False)
        top_features = variances.head(50).index.tolist()
    else:
        top_features = feature_candidates
    
    tradeability_scores, tradeability_metrics = generate_tradeability_scores(
        X=df[top_features].values,
        y=tradeability_y.values,
        timestamps=df['Time'],
        n_folds=TRADEABILITY_N_FOLDS,
        min_train_size=TRADEABILITY_MIN_TRAIN,
        test_size=TRADEABILITY_TEST_SIZE,
        gap_size=outcome_params['max_horizon'],
    )
    
    df['tradeability_score'] = tradeability_scores['tradeability_score'].values
    # Rows never assigned to a test fold will have NaN; default to neutral 0.5
    df['tradeability_score'] = df['tradeability_score'].fillna(0.5)
    print(f"Tradeability score range: [{df['tradeability_score'].min():.4f}, {df['tradeability_score'].max():.4f}]")
    print(f"Tradeability ROC AUC: {tradeability_metrics['overall_roc_auc']:.4f}")
    print(f"Tradeability PR AUC: {tradeability_metrics['overall_pr_auc']:.4f}")
else:
    df['tradeability_score'] = 1.0

Generating tradeability scores...
Processing fold 1/5: train [0:99999], test [100030:150029]
[LightGBM] [Info] Number of positive: 96906, number of negative: 3094
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007814 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11878
[LightGBM] [Info] Number of data points in the train set: 100000, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Processing fold 2/5: train [0:150029], test [150060:200059]
[LightGBM] [Info] Number of positive: 145421, number of negative: 4609
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010343 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11886
[LightGBM] [Info] Number of data points in the train set: 150030, number of used fea

In [ ]:
# Preprocess features and create train / holdout splits
preprocess_ohlcv_args = {
    "target_col": TARGET_COLS + ["tradeability_score"] + (CLASSIFICATION_TARGETS if TRAIN_CLASSIFIER else []),
    "outcomes_col": None,
    "shift": 0,
    "onehot_prefixes": ["OH_"],
    "price_prefixes": ["PR_"],
}

split_idx = int(len(df) * TRAIN_FRACTION)
df_train = df.iloc[:split_idx].copy().reset_index(drop=True)
df_holdout = df.iloc[split_idx:].copy().reset_index(drop=True)

X_train, y_train, scaler, features, _, proc_df_train = preprocess_ohlcv(
    df_train, **preprocess_ohlcv_args, scaler=None, return_df=True,
)
X_holdout, y_holdout, _, _, _, proc_df_holdout = preprocess_ohlcv(
    df_holdout, **preprocess_ohlcv_args, scaler=scaler, return_df=True,
)

# Split target arrays for regression targets
regression_target_count = len(TARGET_COLS)
target_arrays = {
    target: {
        "train": y_train[:, i],
        "holdout": y_holdout[:, i],
    }
    for i, target in enumerate(TARGET_COLS)
}

# Classification target arrays
if TRAIN_CLASSIFIER:
    clf_target_arrays = {
        target: {
            "train": y_train[:, regression_target_count + 1 + i],
            "holdout": y_holdout[:, regression_target_count + 1 + i],
        }
        for i, target in enumerate(CLASSIFICATION_TARGETS)
    }

# Tradeability weight is at index len(TARGET_COLS) (the tradeability_score column)
tradeability_idx = len(TARGET_COLS)
train_weight = None if not USE_TRADEABILITY_WEIGHTING else y_train[:, tradeability_idx]

tune_rows = min(len(X_train), MAX_TUNING_ROWS)
X_tune = X_train[-tune_rows:]
tune_targets = {k: v["train"][-tune_rows:] for k, v in target_arrays.items()}
tune_weight = None if train_weight is None else train_weight[-tune_rows:]

tune_test_size = max(20_000, int(tune_rows * WFO_TEST_FRAC))
tune_min_train = max(100_000, int(tune_rows * WFO_MIN_TRAIN_FRAC))
tune_min_train = min(tune_min_train, tune_rows - tune_test_size - WFO_GAP_ROWS)
while tune_min_train + tune_test_size + WFO_GAP_ROWS > tune_rows and tune_test_size > 10_000:
    tune_test_size = int(tune_test_size * 0.8)
    tune_min_train = min(tune_min_train, tune_rows - tune_test_size - WFO_GAP_ROWS)

wf_folds = walk_forward_folds(
    n_samples=tune_rows,
    min_train_size=tune_min_train,
    test_size=tune_test_size,
    gap_size=max(WFO_GAP_ROWS, outcome_params["max_horizon"]),
    n_folds=WFO_FOLDS,
)

print(f"Train rows   : {len(X_train):,}")
print(f"Holdout rows : {len(X_holdout):,}")
print(f"Tuning rows  : {len(X_tune):,}")
print(f"Walk-forward folds: {len(wf_folds)}")
for i, (tr, va) in enumerate(wf_folds, start=1):
    print(f"  fold {i}: train={len(tr):,} val={len(va):,}")

Train rows   : 449,745
Holdout rows : 49,999
Tuning rows  : 350,000
Walk-forward folds: 4
  fold 1: train=192,500 val=35,000
  fold 2: train=227,560 val=35,000
  fold 3: train=262,620 val=35,000
  fold 4: train=297,680 val=35,000


In [ ]:
# Tune hyper-parameters for each target
tuning_results = {}
best_config_by_target = {}
best_iteration_by_target = {}

for target_name in TARGET_COLS:
    print(f"\nTuning target: {target_name}")
    results_df, best_cfg = tune_lgbm_by_spearman(
        X=X_tune,
        y=tune_targets[target_name],
        folds=wf_folds,
        candidate_params=LGBM_CANDIDATES,
        base_params=LGBM_BASE_PARAMS,
        sample_weight=tune_weight,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        robustness_penalty=ROBUSTNESS_PENALTY,
    )
    tuning_results[target_name] = results_df
    best_config_by_target[target_name] = best_cfg
    best_iteration_by_target[target_name] = int(best_cfg["median_best_iteration"])

    print("Top candidates:")
    display(results_df[[
        "cfg_id", "mean_spearman", "std_spearman", "robust_score",
        "mean_r2", "mean_rmse", "median_best_iteration",
        "learning_rate", "num_leaves", "min_child_samples",
        "subsample", "colsample_bytree", "reg_alpha", "reg_lambda",
    ]].head(5).round(4))

tuning_summary = pd.DataFrame([
    {
        "target": target,
        "best_cfg_id": best_config_by_target[target]["cfg_id"],
        "best_mean_spearman": best_config_by_target[target]["mean_spearman"],
        "best_std_spearman": best_config_by_target[target]["std_spearman"],
        "best_robust_score": best_config_by_target[target]["robust_score"],
        "best_n_estimators": best_iteration_by_target[target],
    }
    for target in TARGET_COLS
]).set_index("target")

print("\nSelected tuning summary:")
display(tuning_summary.round(4))


Tuning target: long_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,2,0.3389,0.0083,0.3368,0.0880,0.7036,136,0.020,127,80,0.85,0.85,0.20,1.0
1,6,0.3386,0.0081,0.3365,0.0882,0.7035,136,0.020,95,50,0.80,0.75,0.00,1.0
2,4,0.3387,0.0087,0.3365,0.0889,0.7032,57,0.050,31,100,0.75,0.75,0.05,5.0
3,3,0.3386,0.0084,0.3365,0.0869,0.7040,217,0.010,255,120,0.90,0.90,0.50,3.0
4,5,0.3384,0.0081,0.3364,0.0883,0.7035,113,0.025,63,40,0.85,0.80,0.10,2.0



Tuning target: short_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,6,0.3341,0.0045,0.3330,0.0816,0.7013,167,0.020,95,50,0.80,0.75,0.00,1.0
1,4,0.3341,0.0051,0.3329,0.0865,0.6992,64,0.050,31,100,0.75,0.75,0.05,5.0
2,1,0.3340,0.0047,0.3328,0.0859,0.6995,102,0.030,63,60,0.80,0.80,0.10,0.5
3,5,0.3339,0.0044,0.3328,0.0844,0.7001,132,0.025,63,40,0.85,0.80,0.10,2.0
4,2,0.3340,0.0049,0.3327,0.0853,0.6997,143,0.020,127,80,0.85,0.85,0.20,1.0



Tuning target: signed_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,4,0.2729,0.0117,0.2700,0.0910,1.1898,47,0.050,31,100,0.75,0.75,0.05,5.0
1,5,0.2729,0.0120,0.2699,0.0924,1.1888,99,0.025,63,40,0.85,0.80,0.10,2.0
2,1,0.2726,0.0116,0.2697,0.0921,1.1891,91,0.030,63,60,0.80,0.80,0.10,0.5
3,2,0.2727,0.0117,0.2697,0.0834,1.1952,91,0.020,127,80,0.85,0.85,0.20,1.0
4,6,0.2725,0.0118,0.2696,0.0912,1.1897,116,0.020,95,50,0.80,0.75,0.00,1.0



Selected tuning summary:


,best_cfg_id,best_mean_spearman,best_std_spearman,best_robust_score,best_n_estimators
target,,,,,
long_quality,2,0.3389,0.0083,0.3368,136
short_quality,6,0.3341,0.0045,0.3330,167
signed_quality,4,0.2729,0.0117,0.2700,47


In [ ]:
# Train final models on full training set and evaluate on holdout
final_models = {}
holdout_results = {}

for target_name in TARGET_COLS:
    best_cfg = best_config_by_target[target_name]
    final_params = {
        **LGBM_BASE_PARAMS,
        **{k: v for k, v in best_cfg.items() if k in LGBM_CANDIDATES[0]},
        "n_estimators": int(best_iteration_by_target[target_name]),
    }

    print(f"Training final model for {target_name} with {final_params['n_estimators']} trees")
    model = LGBMRegressor(**final_params)
    fit_kwargs = {"X": X_train, "y": target_arrays[target_name]["train"]}
    if train_weight is not None:
        fit_kwargs["sample_weight"] = train_weight
    model.fit(**fit_kwargs)

    pred_holdout = model.predict(X_holdout)
    final_models[target_name] = model
    holdout_results[target_name] = reg_metrics(
        target_arrays[target_name]["holdout"], pred_holdout
    )
    holdout_results[target_name]["best_n_estimators"] = int(final_params["n_estimators"])

signed_pred = final_models["signed_quality"].predict(X_holdout)
long_pred = final_models["long_quality"].predict(X_holdout)
short_pred = final_models["short_quality"].predict(X_holdout)
spread_pred = long_pred - short_pred

holdout_results["spread_proxy"] = reg_metrics(
    target_arrays["signed_quality"]["holdout"], spread_pred
)
holdout_results["spread_proxy"]["best_n_estimators"] = int(
    np.median([
        best_iteration_by_target["long_quality"],
        best_iteration_by_target["short_quality"],
    ])
)

holdout_df = pd.DataFrame(holdout_results).T
print("\nHoldout performance:")
display(holdout_df.round(4))

cv_best = pd.DataFrame([
    {
        "target": target,
        "cv_mean_spearman": tuning_summary.loc[target, "best_mean_spearman"],
        "cv_std_spearman": tuning_summary.loc[target, "best_std_spearman"],
        "cv_robust_score": tuning_summary.loc[target, "best_robust_score"],
        "holdout_spearman": holdout_results[target]["Spearman"],
        "holdout_r2": holdout_results[target]["R2"],
    }
    for target in TARGET_COLS
]).set_index("target")

print("\nCV vs holdout:")
display(cv_best.round(4))

Training final model for long_quality with 136 trees


Training final model for short_quality with 167 trees
Training final model for signed_quality with 47 trees

Holdout performance:


,MAE,RMSE,R2,Spearman,best_n_estimators
long_quality,0.4836,0.6719,0.0931,0.3470,136.0
short_quality,0.4752,0.6641,0.0841,0.3360,167.0
signed_quality,0.8683,1.1408,0.0924,0.2735,47.0
spread_proxy,0.8691,1.1410,0.0922,0.2738,151.0



CV vs holdout:


,cv_mean_spearman,cv_std_spearman,cv_robust_score,holdout_spearman,holdout_r2
target,,,,,
long_quality,0.3389,0.0083,0.3368,0.3470,0.0931
short_quality,0.3341,0.0045,0.3330,0.3360,0.0841
signed_quality,0.2729,0.0117,0.2700,0.2735,0.0924


In [ ]:
# Train Classification Models
if TRAIN_CLASSIFIER:
    final_classifiers = {}
    classifier_holdout_results = {}
    
    for target_name in CLASSIFICATION_TARGETS:
        print(f"Training classifier for {target_name}")
        
        clf = LGBMClassifier(
            objective='binary',
            boosting_type='gbdt',
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
            class_weight='balanced',
        )
        
        clf.fit(
            X_train,
            clf_target_arrays[target_name]['train'],
            sample_weight=train_weight,
        )
        
        pred_proba = clf.predict_proba(X_holdout)[:, 1]
        y_true_clf = clf_target_arrays[target_name]['holdout']
        
        classifier_holdout_results[target_name] = {
            'ROC_AUC': float(roc_auc_score(y_true_clf, pred_proba)),
            'PR_AUC': float(average_precision_score(y_true_clf, pred_proba)),
        }
        final_classifiers[target_name] = clf
    
    print("\nClassifier holdout performance:")
    display(pd.DataFrame(classifier_holdout_results).T.round(4))

Training classifier for buy_win


Training classifier for sell_win
Training classifier for signed_win

Classifier holdout performance:


,ROC_AUC,PR_AUC
buy_win,0.6078,0.3169
sell_win,0.5864,0.3031
signed_win,0.5924,0.5835


In [ ]:
# Train Ensemble Models
if ENSEMBLE_SIZE > 1:
    ensemble_models = {target: [] for target in TARGET_COLS}
    
    for target_name in TARGET_COLS:
        best_cfg = best_config_by_target[target_name]
        base_params_for_target = {
            **LGBM_BASE_PARAMS,
            **{k: v for k, v in best_cfg.items() if k in LGBM_CANDIDATES[0]},
        }
        
        for seed in ENSEMBLE_SEEDS[:ENSEMBLE_SIZE]:
            params = {
                **base_params_for_target,
                'random_state': seed,
                'n_estimators': int(best_iteration_by_target[target_name]),
            }
            
            model = LGBMRegressor(**params)
            model.fit(X_train, target_arrays[target_name]['train'])
            ensemble_models[target_name].append(model)
        
        print(f"Trained {len(ensemble_models[target_name])} models for {target_name}")
    
    # Evaluate ensemble on holdout
    ensemble_holdout_results = {}
    for target_name in TARGET_COLS:
        ensemble_preds = np.column_stack([
            m.predict(X_holdout) for m in ensemble_models[target_name]
        ])
        pred_mean = np.mean(ensemble_preds, axis=1)
        ensemble_holdout_results[target_name] = reg_metrics(
            target_arrays[target_name]['holdout'], pred_mean
        )
    
    print("\nEnsemble holdout performance:")
    display(pd.DataFrame(ensemble_holdout_results).T.round(4))
    
    # Compare single vs ensemble
    comparison = pd.DataFrame({
        'single': [holdout_results[t]['Spearman'] for t in TARGET_COLS],
        'ensemble': [ensemble_holdout_results[t]['Spearman'] for t in TARGET_COLS],
    }, index=TARGET_COLS)
    comparison['improvement'] = comparison['ensemble'] - comparison['single']
    print("\nSingle vs Ensemble Spearman:")
    display(comparison.round(4))

Trained 5 models for long_quality
Trained 5 models for short_quality
Trained 5 models for signed_quality

Ensemble holdout performance:


,MAE,RMSE,R2,Spearman
long_quality,0.4833,0.6718,0.0934,0.3470
short_quality,0.4754,0.6639,0.0846,0.3364
signed_quality,0.8684,1.1409,0.0922,0.2736



Single vs Ensemble Spearman:


,single,ensemble,improvement
long_quality,0.3470,0.3470,0.0001
short_quality,0.3360,0.3364,0.0003
signed_quality,0.2735,0.2736,0.0001


In [ ]:
# Trading-style proxy analysis on the signed target
y_true_signed = target_arrays["signed_quality"]["holdout"]
y_pred_signed = signed_pred
thresholds = np.quantile(np.abs(y_pred_signed), np.linspace(0.50, 0.95, 10))

sweep_rows = []
for thr in thresholds:
    pred_sign = np.zeros(len(y_pred_signed), dtype=int)
    pred_sign[y_pred_signed >= thr] = 1
    pred_sign[y_pred_signed <= -thr] = -1
    taken = pred_sign != 0

    if taken.sum() == 0:
        sweep_rows.append({
            "threshold": float(thr),
            "coverage": 0.0,
            "taken": 0,
            "long_count": 0,
            "short_count": 0,
            "sign_hit_rate": np.nan,
            "aligned_signed_mean": np.nan,
            "taken_spearman": np.nan,
        })
        continue

    sign_hit = np.mean(np.sign(y_true_signed[taken]) == pred_sign[taken])
    aligned_signed = np.where(
        pred_sign[taken] == 1, y_true_signed[taken], -y_true_signed[taken]
    )
    sweep_rows.append({
        "threshold": float(thr),
        "coverage": float(taken.mean()),
        "taken": int(taken.sum()),
        "long_count": int((pred_sign == 1).sum()),
        "short_count": int((pred_sign == -1).sum()),
        "sign_hit_rate": float(sign_hit),
        "aligned_signed_mean": float(np.mean(aligned_signed)),
        "taken_spearman": float(safe_spearman(y_true_signed[taken], y_pred_signed[taken])),
    })

threshold_df = pd.DataFrame(sweep_rows)
print("Signed-model threshold sweep:")
display(threshold_df.round(4))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=threshold_df["threshold"],
    y=threshold_df["aligned_signed_mean"],
    mode="lines+markers",
    name="Aligned signed mean",
))
fig.add_trace(go.Scatter(
    x=threshold_df["threshold"],
    y=threshold_df["sign_hit_rate"],
    mode="lines+markers",
    name="Sign hit rate",
))
fig.update_layout(
    title="Trading Proxy vs Confidence Threshold",
    xaxis_title="Absolute prediction threshold",
    yaxis_title="Score",
    height=420,
)
fig.show()

Signed-model threshold sweep:


,threshold,coverage,taken,long_count,short_count,sign_hit_rate,aligned_signed_mean,taken_spearman
0,0.2855,0.50,25000,12573,12427,0.6453,0.4671,0.3578
1,0.3143,0.45,22500,11348,11152,0.6524,0.4857,0.3698
2,0.3427,0.40,20000,10119,9881,0.6620,0.5104,0.3834
3,0.3745,0.35,17500,8872,8628,0.6729,0.5364,0.3967
4,0.4055,0.30,15000,7626,7374,0.6836,0.5585,0.4114
5,0.4336,0.25,12500,6379,6121,0.6938,0.5791,0.4286
6,0.4634,0.20,10000,5137,4863,0.7116,0.6113,0.4483
7,0.4972,0.15,7500,3773,3727,0.7289,0.6393,0.4755
8,0.5395,0.10,5000,2459,2541,0.7552,0.6742,0.5131
9,0.6068,0.05,2500,1163,1337,0.7944,0.7437,0.5706


In [ ]:
# Export the model pack
today = pd.Timestamp.now().strftime("%Y%m%d")
ds_title = DS_NAME.split("/")[-1].split(".")[0]
model_name = "_".join([ds_title, "LightGBM", today, "spearman_prod", "TP" + str(TP_MULT), "SL" + str(SL_MULT)])

model_pack_dir = Path("../ModelWorkbench/ModelPacks")
model_pack_dir.mkdir(parents=True, exist_ok=True)
model_pack_path = model_pack_dir / f"{model_name}_model.pkl"

model_info = {
    "dataset_name": ds_title,
    "dataset_dir": DS_NAME,
    "date_trained": today,
    "model_type": "LightGBMRegressionPack",
    "task": "regression",
    "primary_target": PRIMARY_TARGET,
    "targets": TARGET_COLS,
    "cv_scheme": {
        "type": "expanding_walk_forward",
        "folds": len(wf_folds),
        "gap_rows": max(WFO_GAP_ROWS, outcome_params["max_horizon"]),
        "tuning_rows": int(len(X_tune)),
        "holdout_fraction": TRAIN_FRACTION,
        "robustness_penalty": ROBUSTNESS_PENALTY,
    },
}

pack = {
    "model": final_models[PRIMARY_TARGET],
    "primary_model": final_models[PRIMARY_TARGET],
    "aux_models": {k: v for k, v in final_models.items() if k != PRIMARY_TARGET},
    "model_class": final_models[PRIMARY_TARGET].__class__,
    "model_class_source": inspect.getsource(final_models[PRIMARY_TARGET].__class__),
    "model_params": {
        target: {
            **LGBM_BASE_PARAMS,
            **{k: v for k, v in best_config_by_target[target].items() if k in LGBM_CANDIDATES[0]},
            "n_estimators": int(best_iteration_by_target[target]),
        }
        for target in TARGET_COLS
    },
    "model_info": model_info,
    "features": features,
    "selected_features": selected_features if "selected_features" in globals() else None,

    "feature_count": X_train.shape[1],
    "feature_function": FEATURES,
    "feature_function_source": inspect.getsource(FEATURES),
    "preprocess_function": preprocess_ohlcv,
    "preprocess_function_source": inspect.getsource(preprocess_ohlcv),
    "preprocess_args": preprocess_ohlcv_args,
    "scaler": scaler,
    "label_function": calculate_trade_outcomes_capped,
    "label_function_source": inspect.getsource(calculate_trade_outcomes_capped),
    "label_params": None,
    "regime_params": regime_params,
    "outcome_params": outcome_params,
    "trading_hours": None,
    "input_shape": None,
    "data_split": {
        "train_rows": len(X_train),
        "holdout_rows": len(X_holdout),
        "train_fraction": TRAIN_FRACTION,
        "max_tuning_rows": MAX_TUNING_ROWS,
    },
    "validation": {
        "cv_summary": tuning_summary.to_dict(orient="index"),
        "cv_results": {
            target: tuning_results[target].head(10).to_dict(orient="records")
            for target in TARGET_COLS
        },
        "holdout": {
            target: {k: float(v) for k, v in holdout_results[target].items()}
            for target in holdout_results
        },
        "threshold_sweep": threshold_df.to_dict(orient="records"),
    },
}

# Add classifiers to pack if trained
if TRAIN_CLASSIFIER:
    pack['classifiers'] = final_classifiers
    pack['classifier_validation'] = classifier_holdout_results

# Add ensemble models to pack if trained
if ENSEMBLE_SIZE > 1:
    pack['ensemble_models'] = ensemble_models
    pack['ensemble_size'] = ENSEMBLE_SIZE
    pack['ensemble_seeds'] = ENSEMBLE_SEEDS[:ENSEMBLE_SIZE]
    pack['ensemble_validation'] = ensemble_holdout_results

with open(model_pack_path, "wb") as fh:
    pickle.dump(pack, fh)

print(f"Saved model pack to {model_pack_path}")

Saved model pack to ../ModelWorkbench/ModelPacks/US500_M1_520weeks_LightGBM_20260705_spearman_prod_TP3.0_SL2.0_model.pkl


In [ ]:
# Model importance snapshot for the primary deployment model
importance_df = pd.DataFrame({
    "feature": features,
    "importance": final_models[PRIMARY_TARGET].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

display(importance_df.head(30))

fig = go.Figure(go.Bar(
    x=importance_df.head(30)["importance"][::-1],
    y=importance_df.head(30)["feature"][::-1],
    orientation="h",
))
fig.update_layout(
    title=f"Top 30 Feature Importances - {PRIMARY_TARGET}",
    height=700,
    xaxis_title="Gain / split importance",
    yaxis_title="Feature",
)
fig.show()

,feature,importance
0,fl_close_loc,194
1,fl_hl_range_atr,138
2,fl_atr_ratio_14,133
3,fl_H_rel,63
4,fl_hour_sin,60
5,fl_price_loc_256,49
6,fl_hour_cos,48
7,fl_pct_from_low_60,48
8,fl_body_atr,42
9,fl_rv_ratio_20_60,42
